In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [2]:
baseurl = "https://loveandflair.com/collections/activewear-1"

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml",
    "Connection": "keep-alive"
}

In [4]:
url = baseurl + "?page=1"
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")
print("STATUS:", response.status_code)

STATUS: 200


In [5]:
pages = set()
for a in soup.find_all("a"):
    href = a.get("href")
    if href and "page=" in href:
        try:
            page_num = int(href.split("page=")[-1])
            pages.add(page_num)
        except:
            pass
print("PAGES FOUND:", sorted(pages))
print("TOTAL PAGES:", max(pages) if pages else 1)

PAGES FOUND: [2, 3, 6]
TOTAL PAGES: 6


In [6]:
total_pages = max(pages) if pages else 1
print("FINAL PAGE COUNT:", total_pages)

FINAL PAGE COUNT: 6


In [7]:
soup = BeautifulSoup(response.text, "html.parser")
products_test = soup.find_all("li", class_="collection-product-card")
print("PRODUCTS FOUND (PAGE 1):", len(products_test))
if len(products_test) > 0:
    print(products_test[0].prettify())

PRODUCTS FOUND (PAGE 1): 20
<li class="collection-product-card quickview">
 <link as="style" href="//loveandflair.com/cdn/shop/t/65/assets/section-main-product.css?v=151365208571903715461701879102" onload="this.onload=null;this.rel='stylesheet'" rel="preload"/>
 <link href="//loveandflair.com/cdn/shop/t/65/assets/component-deferred-media.css?v=133393484956369264981701879092" media="all" rel="stylesheet"/>
 <link as="style" href="//loveandflair.com/cdn/shop/t/65/assets/quick-add.css?v=145050549743491949031701879099" onload="this.onload=null;this.rel='stylesheet'" rel="preload"/>
 <script defer="defer" src="//loveandflair.com/cdn/shop/t/65/assets/product-form.js?v=68147277516988630661701879098">
 </script>
 <script defer="defer" src="//loveandflair.com/cdn/shop/t/65/assets/quick-add.js?v=160922982854406242821720433695">
 </script>
 <div class="card-wrapper js-color-swatches-wrapper" data-product="gush-top-in-orange">
  <span class="visually-hidden">
   Gush Top in Orange
  </span>
  <div

In [8]:
total_products = 0
for page in range(1, 50):
    url = f"{baseurl}?page={page}"
    r = requests.get(url, headers=headers)
    if r.status_code == 403:
        print("Blocked at page", page)
        break
    soup = BeautifulSoup(r.text, "html.parser")
    products = soup.find_all("li", class_="collection-product-card")
    count = len(products)
    print(f"Page {page} → {count} products")
    if count == 0:
        break
    total_products += count
print("\nTOTAL PRODUCTS:", total_products)

Page 1 → 20 products
Page 2 → 20 products
Page 3 → 20 products
Page 4 → 20 products
Page 5 → 20 products
Page 6 → 12 products
Page 7 → 0 products

TOTAL PRODUCTS: 112


In [9]:
data = []
for page in range(1, 7):
    if page == 4:
        print("\nWAITING.....\n")
        time.sleep(60)
    print("\n==============================")
    print("SCRAPING PAGE:", page)
    print("==============================")
    url = f"{baseurl}?page={page}"
    response = requests.get(url, headers=headers)
    print("STATUS:", response.status_code)
    print("URL:", response.url)
    soup = BeautifulSoup(response.text, "html.parser")
    products = soup.find_all("li", class_="collection-product-card")
    print("PRODUCTS FOUND:", len(products))
    if len(products) == 0:
        print("No products found")
        continue
    for product in products:
        try:
            name = product.find(class_='card__title').text.strip()
        except:
            name = "N/A"
        try:
            price = product.find(class_='price-item--regular').text.strip()
        except:
            price = "N/A"
        try:
            stock_tag = product.find(class_='sold-out')
            stock = "Sold Out" if stock_tag else "In Stock"
        except:
            stock = "N/A"
        try:
            color = product.find(class_='color-swatch')['title']
        except:
            color = "N/A"
        try:
            product_link = product.find(class_='card-product__link')['href']
            full_link = "https://loveandflair.com" + product_link
            product_response = requests.get(full_link, headers=headers)
            product_soup = BeautifulSoup(product_response.text, 'html.parser')
            size_tags = product_soup.find_all(class_='link-hover-line-outer')
            size_list = []
            for size in size_tags:
                span = size.find('span', class_='uppercase')
                if span:
                    text = span.text.strip()
                    if text not in size_list:
                        size_list.append(text)
            sizes = ", ".join(size_list)
        except:
            sizes = "N/A"
        data.append({
            "Name": name,
            "Price": price,
            "Stock": stock,
            "Color": color,
            "Size": sizes
        })
        print(name, "|", price, "|", stock, "|", color, "|", sizes)


SCRAPING PAGE: 1
STATUS: 200
URL: https://loveandflair.com/collections/activewear-1?page=1
PRODUCTS FOUND: 20
Gush Top in Orange | IDR 399.000 | In Stock | Orange | XS, S, M, L
Ventra Zipper Tank Top in Orange | IDR 499.000 | In Stock | Orange | XS, S, M, L
Motive Shorts in White | IDR 459.000 | In Stock | White | XS, S, M, L
Motive Shorts in Black | IDR 459.000 | In Stock | Black | XS, S, M, L
Manifest Ribbed White Bra | IDR 439.000 | In Stock | White | XS, S, M, L
Route Pocket Tank Top in Orange | IDR 480.000 | In Stock | Orange | XS, S, M, L
Victory Jacket in White | IDR 599.000 | In Stock | White | XS, S, M, L
Route Pocket Tank Top in White | IDR 480.000 | In Stock | White | XS, S, M, L
Ventra Zipper Tank Top in Midnight Navy | IDR 499.000 | In Stock | Navy | XS, S, M, L
Ventra Zipper Tank Top in White | IDR 499.000 | In Stock | White | XS, S, M, L
Dual Top in White | IDR 399.000 | In Stock | White | XS, S, M, L
Lux High Impact Leggings in Black | IDR 509.000 | In Stock | Black | 

In [10]:
df = pd.DataFrame(data, columns=["Name", "Price", "Stock", "Color", "Size"])
print("TOTAL PRODUCTS SCRAPED:", len(df))
df.head(10)

TOTAL PRODUCTS SCRAPED: 112


,Name,Price,Stock,Color,Size
0,Gush Top in Orange,IDR 399.000,In Stock,Orange,"XS, S, M, L"
1,Ventra Zipper Tank Top in Orange,IDR 499.000,In Stock,Orange,"XS, S, M, L"
2,Motive Shorts in White,IDR 459.000,In Stock,White,"XS, S, M, L"
3,Motive Shorts in Black,IDR 459.000,In Stock,Black,"XS, S, M, L"
4,Manifest Ribbed White Bra,IDR 439.000,In Stock,White,"XS, S, M, L"
5,Route Pocket Tank Top in Orange,IDR 480.000,In Stock,Orange,"XS, S, M, L"
6,Victory Jacket in White,IDR 599.000,In Stock,White,"XS, S, M, L"
7,Route Pocket Tank Top in White,IDR 480.000,In Stock,White,"XS, S, M, L"
8,Ventra Zipper Tank Top in Midnight Navy,IDR 499.000,In Stock,Navy,"XS, S, M, L"
9,Ventra Zipper Tank Top in White,IDR 499.000,In Stock,White,"XS, S, M, L"
